# 03 Futures Basis

Purpose: inspect the processed futures basis table, annualized basis and rich/fair/cheap classification.

Data source: `data/processed/futures_basis.parquet` from public/delayed MOEX ISS FORTS and TQBR spot data.


## Methodology Summary

Basis = futures price minus spot price. Annualized basis scales percentage basis by days to expiry. The classification is a carry screen, not an arbitrage signal.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from russian_markets_lab.data.io import read_processed_dataset, read_processed_metadata

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

def load_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    df = read_processed_dataset(name)
    metadata = read_processed_metadata(name)
    print(f'{name}: {len(df)} rows, {len(df.columns)} columns')
    if metadata:
        print('generated_at:', metadata.get('generated_at'))
        print('source:', metadata.get('source'))
    else:
        print('metadata: missing')
    return df, metadata

def show_missing(name: str) -> None:
    print(f'No processed data found for {name}. Run the corresponding CLI pipeline first.')


In [ ]:
basis, metadata = load_dataset('futures_basis')
if basis.empty:
    show_missing('futures_basis')
else:
    display(basis.head(30))


## Annualized Basis Chart


In [ ]:
if not basis.empty and {'futures_secid', 'annualized_basis'}.issubset(basis.columns):
    chart_df = basis.dropna(subset=['annualized_basis']).sort_values('annualized_basis')
    fig = px.bar(chart_df, x='futures_secid', y='annualized_basis', color='signal' if 'signal' in chart_df.columns else None, title='Annualized basis by futures contract')
    fig.show()


## Rich/Fair/Cheap Screen


In [ ]:
if not basis.empty and 'signal' in basis.columns:
    display(basis.groupby('signal', dropna=False).size().rename('contracts').reset_index())


## Key Observations

Record contract mappings, liquidity filters and any unusual basis deviations after execution.


## Limitations And Disclaimer

Mapping futures to spot is best effort. This notebook makes no arbitrage claim and provides no trading advice.
